In [ ]:
import os
from langgraph.prebuilt import create_react_agent
from langhchain.agents import Tool
from langchai.tools import WikipediaQueryRun
from lamgchain.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OpenAIEmbeddings
from langchain.text_splitters import RecursiveCharacterTextSplitter
from langgraph.grap import StateGraph
from typing import Annotated, TypedDict, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph.message import add_messages
from langhchain.loaders import WebbasedLoader

In [ ]:
#create retriver tool

#load content from blog
docs= WebbasedLoader("https://www.oreilly.com/library/view/learning-python-5th/9781449355722/").load()
spiltter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = spiltter.split_documents(docs)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [ ]:
def retriever_tool(query:str) -> str:
    print("Calling retriever tool")
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

In [ ]:
retriver_tool = Tool(
    name="RAG Retriever Tool",
    description="A tool to retrieve relevant information from a vector store based on a query.",
    func=retriever_tool)

In [ ]:
## wiki tool
wiki = WikipediaAPIWrapper()
def wiki_tool(query:str) -> str:
    print("Calling wiki tool")
    return wiki.run(query)

wiki_tool = Tool(
    name="Wikipedia Tool",  
    description="A tool to query Wikipedia for information.",
    func=wiki_tool
)

In [ ]:
tools=[retriver_tool, wiki_tool]

#create agent
react_node = create_react_agent(llm,tools)

In [ ]:
#3 agent state

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("react_node", react_node)
builder.set_initial_node("react_node")
builder.add_edge("react_node", END )

graph= builder.compile()